In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Create data
# Create data with a relationship similar to y = 2x + 1
X = torch.randn(100, 1) * 10 # 100 samples, 1 feature
noise = torch.randn(100, 1)
y = 2 * X + 1 + noise

print("X shape:", X.shape)
print("y shape:", y.shape)

class CustomDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
  # Return the total length of the dataset
    return len(self.X)

  def __getitem__(self, idx):
  # Return the data sample (X, y) corresponding to idx as a tuple
    sample = self.X[idx], self.y[idx]
    return sample

# Create a Dataset object
dataset = CustomDataset(X, y)

# Test if it works
print("Total number of samples:", len(dataset))
first_sample = dataset[0]
print("First sample (X, y):", first_sample)

# 1. Model definition (review of morning session content)
class LinearRegressionModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear = nn.Linear(in_features=1, out_features=1)

  def forward(self, x):
    return self.linear(x)

# 2. Hyperparameter setting
learning_rate = 0.01
num_epochs = 100
batch_size = 10

# 3. DataLoader creation
# Wraps the Dataset to bundle and mix data in batch size units.
data_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True)

# 4. Instantiate model, loss function, and optimizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = LinearRegressionModel().to(device)
loss_function = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
#optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


print("DataLoader, Model, Loss, Optimizer are ready.")
print("--- Training Start ---")
# Training the entire dataset by repeating num_epochs
for epoch in range(num_epochs):
  # DataLoader extracts data in batch size for each iteration.
  for X_batch, y_batch in data_loader:
    # Move data to the same device as the model
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

    # 1. Forward
    predictions = model(X_batch)

    # 2. Calculate loss
    loss = loss_function(predictions, y_batch)

    # 3. PyTorch standard 3-step learning
    optimizer.zero_grad() # Initialize gradient
    loss.backward() # Backpropagation
    optimizer.step() # Update parameters

  # Output intermediate results every 10 epochs
  if (epoch + 1) % 10 == 0:
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("--- Training Finished ---")

# Check the learned parameters (since y = 2x + 1, weight should be close to 2 and bias should be close to 1)
# Since model.parameters() is a generator, convert it to a list and check
trained_params = list(model.parameters())
trained_weight = trained_params[0].item()
trained_bias = trained_params[1].item()

print(f"Trained Weight: {trained_weight:.4f}")
print(f"Trained Bias: {trained_bias:.4f}")